# AN5 — Engagement Sensitivity

This notebook orchestrates the canonical AN5 SQL files. SQL remains the single source of truth under `sql/04_analytics/`; this notebook contains no duplicated query text.

AN5 compares R0, R1, and R2 as metric-only scenarios. It does not modify RAW or STAGING and does not approve an engagement KPI by itself.


## 1. Execution scope and safeguards

- Use the existing PostgreSQL database `oulad_analytics`.
- Confirm the connection is read-only before enabling execution.
- Run one AN5 query at a time; do not use Run All.
- Each query scans the large `student_vle` table and uses a 600,000 ms statement timeout.
- Review every result set, denominator, row count, and truncation flag before proceeding.
- Do not modify RAW, QUALITY, or STAGING. No persistent database objects are required.
- A saved notebook output is evidence of execution; an unexecuted cell or prepared SQL file is not.


## 2. Environment and read-only preflight

The preflight verifies the database identity, transaction read-only settings, and configured timeouts. It does not execute an AN5 analytical query.


In [ ]:
from IPython.display import display
from src.analytics import db

connection = db.connect_read_only(statement_timeout_ms=600000)
try:
    with connection.cursor() as cursor:
        cursor.execute("""
            SELECT current_database() AS database_name,
                   current_setting('transaction_read_only') AS transaction_read_only,
                   current_setting('default_transaction_read_only') AS default_transaction_read_only,
                   current_setting('statement_timeout') AS statement_timeout,
                   current_setting('lock_timeout') AS lock_timeout
        """)
        preflight = cursor.fetchone()
        preflight_columns = [column.name for column in cursor.description]
finally:
    connection.rollback()
    connection.close()

preflight_result = dict(zip(preflight_columns, preflight))
display(preflight_result)
PREFLIGHT_PASSED = (
    preflight_result['database_name'] == 'oulad_analytics'
    and preflight_result['transaction_read_only'] == 'on'
    and preflight_result['default_transaction_read_only'] == 'on'
)
print('AN5 preflight:', 'PASS' if PREFLIGHT_PASSED else 'BLOCKED')


## 3. Explicit live-execution authorization

After reviewing the preflight output, deliberately change the next value to `True` to authorize read-only AN5 query execution in this notebook session. The default remains disabled.


In [ ]:
LIVE_AN5_AUTHORIZED = False
query_results = {}
review_flags = {}
EXPECTED_RESULT_SETS = {'AN5.01': 3, 'AN5.02': 1, 'AN5.03': 1}

def execute_an5(query_id, statement_timeout_ms=600000):
    if not PREFLIGHT_PASSED:
        print(f'{query_id}: NOT EXECUTED — read-only preflight is not satisfied.')
        return None
    if not LIVE_AN5_AUTHORIZED:
        print(f'{query_id}: NOT EXECUTED — explicit AN5 authorization is disabled.')
        return None
    if query_id in query_results:
        print(f'{query_id}: NOT EXECUTED — this query already has an in-memory result. Review or deliberately reinitialize first.')
        return None
    result = db.run_query(query_id, statement_timeout_ms=statement_timeout_ms)
    expected = EXPECTED_RESULT_SETS[query_id]
    if result['result_set_count'] != expected:
        received = result['result_set_count']
        raise RuntimeError(f'{query_id}: expected {expected} result sets, received {received}. Stop and inspect the SQL.')
    query_results[query_id] = result
    print({
        'query_id': query_id,
        'result_sets': result['result_set_count'],
        'elapsed_seconds': result['elapsed_seconds'],
        'statement_timeout_ms': result['statement_timeout_ms'],
        'truncation_by_result_set': [item['display_truncated'] for item in result['result_sets']],
    })
    for item in result['result_sets']:
        print(f"Result set {item['statement_number']} — {item['rows_displayed']} displayed rows; truncated={item['display_truncated']}")
        display(item['data'])
    return result

def record_an5_review(query_id):
    result = query_results.get(query_id)
    if result is None:
        print(f'{query_id}: REVIEW BLOCKED — no retained output.')
        return False
    if any(item['display_truncated'] for item in result['result_sets']):
        print(f'{query_id}: REVIEW BLOCKED — one or more result sets were truncated.')
        return False
    response = input(f"Enter REVIEWED only after the outputs and interpretation have been discussed for {query_id}: ").strip()
    review_flags[query_id] = response == 'REVIEWED'
    print(f"{query_id} review:", 'RECORDED' if review_flags[query_id] else 'NOT RECORDED')
    return review_flags[query_id]


## 4. AN5.01 — Scenario and source-grain reconciliation

This query returns three result sets: the repeated-key profile, R0/R1/R2 row-count bases, and registered-population/activity-key reconciliation. Review all three before proceeding.


In [ ]:
result_an501 = execute_an5('AN5.01', statement_timeout_ms=600000)


In [ ]:
review_an501 = record_an5_review('AN5.01')


## 5. AN5.02 — Student-presentation activity sensitivity

This query compares scenario-specific click volume, student-active-day pairs, student-active-site pairs, and activity coverage using the same registered student-presentation denominator. The overall row is returned alongside module-presentation rows.


In [ ]:
if review_flags.get('AN5.01', False):
    result_an502 = execute_an5('AN5.02', statement_timeout_ms=600000)
else:
    result_an502 = None
    print('AN5.02: NOT EXECUTED — AN5.01 review gate is closed.')


In [ ]:
review_an502 = record_an5_review('AN5.02')


## 6. AN5.03 — Temporal sensitivity

This query separates pre-course, in-course, and post-course activity and distinguishes registered from unmatched activity keys. Negative relative days remain visible; their cause is not inferred.


In [ ]:
if review_flags.get('AN5.02', False):
    result_an503 = execute_an5('AN5.03', statement_timeout_ms=600000)
else:
    result_an503 = None
    print('AN5.03: NOT EXECUTED — AN5.02 review gate is closed.')


In [ ]:
review_an503 = record_an5_review('AN5.03')


## 7. Evidence review and exit gate

AN5 is not closed merely because all three SQL files ran. Review the R0/R1/R2 differences, common denominators, row-count invariants, join matches, temporal buckets, and assumptions. Record the accepted activity definition or keep AN6 gated. No KPI, causal claim, or predictive feature is approved by this notebook alone.
